In [2]:
!pip install quandl
!pip install PyPortfolioOpt==1.5.5
!pip install arch
!pip install numpy

In [3]:
!pip install scikit-learn==1.2.2
import sklearn
import pypfopt
print(sklearn.__version__)
print(pypfopt.__version__)

  error: subprocess-exited-with-error
  
  × Preparing metadata (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> [33 lines of output]
      Partial import of sklearn during the build process.
      Traceback (most recent call last):
        File "C:\Users\ysnow\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 353, in <module>
          main()
        File "C:\Users\ysnow\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\ysnow\anaconda3\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 149, in prepare_metadata_for_build_wheel
          return hook(metadata_directory, config_settings)
                 ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
        File "C:\Users\ysnow\AppData\Local\Temp\pip-build-e

  Using cached scikit-learn-1.2.2.tar.gz (7.3 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'error'
1.4.2
1.5.5


In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime as dt
import datetime
import quandl
import math
import pypfopt
import random
from statsmodels.stats.diagnostic import het_arch
from arch import arch_model
from pypfopt import EfficientFrontier,objective_functions
from sklearn.decomposition import PCA
from sklearn.mixture import GaussianMixture
from sklearn.cluster import KMeans
from sklearn import metrics
from copy import deepcopy

In [5]:
pypfopt.__version__

'1.5.5'

In [6]:
!pip install pandas_datareader

In [7]:
!pip install yfinance==0.2.40
from pandas_datareader import data as pdr
import yfinance as yf
yf.pdr_override()

yfinance: pandas_datareader support is deprecated & semi-broken so will be removed in a future verison. Just use yfinance.


In [8]:
!pip install tslearn
from tslearn.metrics import dtw  # For DTW distance
from scipy.cluster.hierarchy import linkage, fcluster  # Hierarchical clustering
from sklearn.metrics import davies_bouldin_score  # DB Index

In [9]:
def modify_date(date):
    return date[:10]

In [10]:
def data_cleaning(data):
    if (data.isna()).any().any():
        missing = np.where(data.isna())
        rows=missing[0]
        cols=missing[1]
        print(rows)
        for row in range(len(rows)):
            col=cols[row]
            if row == len(data)-1:
                data.iloc[row,col]=data.iloc[row-1,col]**2/data.iloc[row-2,col]
            elif row ==0:
                data.iloc[row,col]=data.iloc[row+1,col]**2/data.iloc[row+2,col]
            else:
                data.iloc[row,col]=math.sqrt(data.iloc[row-1,col]*data.iloc[row+1,col])
        print(data)

In [11]:
def get_price_years_ago(data, date,time):
    year_ago=date-datetime.timedelta(days=int(time*365))
    price_year_ago=data[data.index<=year_ago]
    return price_year_ago

In [12]:
def max_drawdown_absolute(returns):
    r = returns.add(1).cumprod()
    dd = r.div(r.cummax()).sub(1)
    mdd = dd.min()
    end = dd.argmin()
    start = r.iloc[:end].argmax()
    return abs(mdd), start, end

In [13]:
def fit_garch(data,p,q):
    lag = max(p,q)
    length =res.shape[0]
    am = arch_model(res, vol='Garch', p=1, o=0, q=1, dist='Normal')
    split_date = one_year_monthly_return.index.values[lag-1]
    results = am.fit(last_obs=split_date)
    forecasts = results.forecast(horizon=1, start = length-lag, method='simulation', simulations=1000)
    sims = forecasts.variance
    std = np.sqrt(np.sum(sims)*12/lag)
    return std

In [14]:
def post_allocation(dfMVOweight):

    dfMVOweight['equal']=1/len(dfMVOweight)
    numofstocks=len(dfMVOweight)
    allocationweight=allocation.loc[numofstocks,:]
    dfMVOweight['proportion']=''
    dfMVOweight['adjusted proportion']=''
    dfMVOweight['direct']=''
    residual=0
    for i in range(numofstocks):
        dfMVOweight.iloc[i,4]=dfMVOweight.iloc[i,1]/np.sum(dfMVOweight['PR_SR'])
        if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
            dfMVOweight.iloc[i,5]=allocationweight[0]
        else:
            dfMVOweight.iloc[i,5]=(dfMVOweight.iloc[i,4]+residual/(numofstocks-i))
        residual+=dfMVOweight.iloc[i,4]-dfMVOweight.iloc[i,5]
    for i in range(int(allocationweight[1])):
        dfMVOweight.iloc[i,6]=allocationweight[0]
    for i in range(int(allocationweight[1]),int(allocationweight[1]+allocationweight[3])):
        dfMVOweight.iloc[i,6]=allocationweight[2]
    return dfMVOweight

In [15]:
def output(weight):
    if stock_type == "S&P100":
      OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

    elif stock_type == "S&P500":
      OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

    stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]
    OEX.to_csv("oex.csv")
    injected_result = pd.DataFrame(index = stock_price.index )
    S1=np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
    portfolio_value = stock_price@S1.T
    injected_result["portfolio_value"]=portfolio_value
    S2 = 10000/float(OEX[0])
    injected_result["oex"]=S2*OEX
    return injected_result

# Step 1 - Preselection

In [16]:
cd C:\Users\ysnow\OneDrive\Desktop\Black-Swan-Research-main\data\SwanBeta

C:\Users\ysnow\OneDrive\Desktop\Black-Swan-Research-main\data\SwanBeta


- beta is a measure of a stock’s sensitivity to market movements
- PR is 1-avg(swan beta)

In [17]:
# Preselection parameters set up
method = "dtw"          # or avg or any other method
stock_type = "S&P100"
end_swan = "Swan8"         #equals the last swan before the swan we are going to predict
start_swan="Swan4"
start = dt(2019,1,1)
end = dt(2020,2,15)
selected_stock_number = 22     #number of stock preselected
start_swan_date = dt(2020,2,15)
end_swan_date = dt(2021,3,23)
w1 = 0.17
w2 = 0.17
MA_days = 30
N = 7 # This parameter is set to help classification method like K-means

In [18]:
#read all stocks and their swan beta
if stock_type == "S&P100":
    portfolio_beta = pd.read_excel("All_SwanBeta100.xlsx",index_col=0)
else:
    portfolio_beta = pd.read_excel("All_SwanBeta500.xlsx",index_col=0)

In [19]:
# extract beta data for specified swan events
selected_swans_var=portfolio_beta.loc[:,start_swan:end_swan]
portfolio_beta =portfolio_beta[selected_swans_var.T.isnull().sum()<selected_swans_var.shape[1]-1]

In [20]:
portfolio_beta

,Swan -4,Swan -3,Swan -2,Swan -1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
Ticker,,,,,,,,,,,,,,
SO,NaN,NaN,NaN,0.789771,0.735961,-0.130115,-0.265152,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1.173178,0.264469
DUK,NaN,NaN,-0.076048,0.180670,0.400895,-0.126448,-0.519440,1.580780,0.771715,-0.322795,0.233695,-0.356252,1.165454,0.534699
NEE,0.982309,0.253298,-0.134131,0.609003,0.579708,0.166846,-0.420656,0.414536,0.503895,0.415370,0.192031,-0.026634,1.062981,0.710514
VZ,NaN,NaN,NaN,-0.071253,0.547390,0.059871,0.109521,0.905674,0.332333,0.064046,0.717548,0.060183,0.430888,1.036945
EXC,1.101894,0.665427,-0.825250,1.115449,0.634396,0.174697,-0.975565,0.519666,0.794675,0.263392,0.605247,-0.123437,1.254518,0.353268
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
JPM,NaN,NaN,0.873322,1.666279,1.196511,2.138915,1.576224,1.649018,1.453522,1.887942,1.012903,0.969234,1.348489,1.245036
BAC,2.119191,1.270985,0.770965,1.317948,0.992028,2.385911,1.731738,0.594107,2.142401,2.667371,1.184258,1.192922,1.509426,1.256673
MS,NaN,NaN,NaN,NaN,NaN,NaN,2.109127,1.441339,0.770028,2.806836,1.528749,1.047092,1.605162,0.793801


In [21]:
# correlation matrix
corr=portfolio_beta.corr()
selected_corr=corr.loc["Swan1":end_swan,"Swan1":end_swan]
corr1={}
for i in range(selected_corr.shape[1]):
    for j in range(i+1,selected_corr.shape[0]):
        selected_corr.iloc[j,i]=np.nan # By setting the lower triangular part of the correlation matrix to NaN, it avoids redundant comparisons when averaging

In [22]:
selected_corr

,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8
Swan1,1.0,0.003789,0.101347,0.057327,-0.000547,0.083689,0.085222,0.058642
Swan2,NaN,1.000000,0.583145,0.360109,0.432200,0.445509,0.039317,0.367065
Swan3,NaN,NaN,1.000000,0.157751,0.283951,0.423298,0.225053,0.190724
Swan4,NaN,NaN,NaN,1.000000,-0.002432,0.184384,0.126664,0.249096
Swan5,NaN,NaN,NaN,NaN,1.000000,0.374752,0.324799,0.214341
Swan6,NaN,NaN,NaN,NaN,NaN,1.000000,0.265093,0.418944
Swan7,NaN,NaN,NaN,NaN,NaN,NaN,1.000000,0.141909
Swan8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.000000


In [23]:
avg=selected_corr.mean(axis=1)
weight_swan = avg/sum(avg)
selected_beta=portfolio_beta.loc[:,start_swan:end_swan].fillna(0)
#portfolio_beta["corr_beta"]=selected_beata@weight_swan
#portfolio_beta=portfolio_beta.sort_values(by=["corr_beta"])

Dynamic Time Warping (DTW) is a method for measuring similarity between time series data that may vary in speed or alignment.

- Use DTW to calculate a distance matrix between all stocks based on their swan beta values.
- Apply hierarchical clustering on the distance matrix to group similar stocks.
- Select the 22 most representative stocks from different clusters.

In [24]:
# basic dtw method between two sequences
def dtw_single(sequence_a, sequence_b):
    len_a = len(sequence_a)
    len_b = len(sequence_b)
    
    # Create distance matrix using numpy for efficiency
    distance_matrix = np.abs(np.array(sequence_a)[:, np.newaxis] - np.array(sequence_b))
    
    # Initialize DP matrix with infinity padding
    dp = np.full((len_a+1, len_b+1), np.inf)
    dp[0, 0] = 0
    
    # Fill DP matrix
    for i in range(1, len_a+1):
        for j in range(1, len_b+1):
            cost = distance_matrix[i-1, j-1]
            dp[i, j] = cost + min(dp[i-1, j],    # insertion
                                  dp[i, j-1],    # deletion
                                  dp[i-1, j-1])  # match
    
    # Backtrack to find warping path
    path = []
    i, j = len_a, len_b
    while i > 0 or j > 0:
        path.append((i-1, j-1))
        if i == 0:
            j -= 1
        elif j == 0:
            i -= 1
        else:
            min_val = min(dp[i-1, j-1], dp[i-1, j], dp[i, j-1])
            if min_val == dp[i-1, j-1]:
                i -= 1
                j -= 1
            elif min_val == dp[i-1, j]:
                i -= 1
            else:
                j -= 1
                
    path = np.array(path[::-1])  # Reverse and convert to numpy array
    return dp[len_a, len_b], path

In [ ]:
if method == "gamma":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    portfolio_beta["avg_sd"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)+portfolio_beta.loc[:,start_swan:end_swan].std(axis=1,ddof=0)
    portfolio_beta=portfolio_beta.sort_values(by=["avg_sd"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]
elif method =="average":
    portfolio_beta["avg"]=portfolio_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #Select 18 stocks
    portfolio_beta=portfolio_beta.sort_values(by=["avg"])
    preselected_stock = portfolio_beta.iloc[:selected_stock_number,:]

elif method == "K_means":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    Kmean = KMeans(n_clusters=N,random_state=1)#set the random_state to keep modeling the same every run
    Kmean.fit(selected_beta)
    ct = Kmean.cluster_centers_
    labels = Kmean.labels_
    current_dbscore=metrics.davies_bouldin_score(selected_beta, labels)
    print(current_dbscore)
    selected_beta["Group"]=Kmean.labels_
    print(selected_beta['Group'])
    res = ct.mean(axis=1)
    res [res<0]=0

    min_group = np.where(res==min(res))
    for i in range(N):
        Y_predic[selected_beta["Group"] == i] = res [i]
    selected_beta ["avg"]=Y_predic.values
    selected_beta=selected_beta.sort_values(by=["avg"])
    first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
    if first_group.shape[0] <10:
        second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
        preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
    else:
        preselected_stock = first_group

# Use results from multiple clusters
elif method=="K_means_multiple":
    selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
    Y_predic = selected_beta[end_swan].copy()
    seed_dict={}

    # Run 1000 trials with davies bouldin index
    for r in range(1,1000):
      Kmean = KMeans(n_clusters=N,random_state=r,n_init=10)
      Kmean.fit(selected_beta)
      labels = Kmean.labels_
      # Evaluate the davies-bouldin score
      dbscore=metrics.davies_bouldin_score(selected_beta, labels)
      seed_dict[r]=dbscore
      sorted_dict=sorted(seed_dict.items(), key=lambda x:x[1])

    # Put all stocks into a stock list
    stock_list=[]
    max_length=0
    for index in range(0,5):
      selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()
      Kmean = KMeans(n_clusters=N,random_state=sorted_dict[index][0],n_init=10) # rerun k means with the best seed
      Kmean.fit(selected_beta)
      ct = Kmean.cluster_centers_
      Kmean.labels_
      selected_beta["Group"]=Kmean.labels_
      res = ct.mean(axis=1)
      res [res<0]=0
      min_group = np.where(res==min(res))
      for i in range(N):
          Y_predic[selected_beta["Group"] == i] = res [i]
      selected_beta ["avg"]=Y_predic.values
      selected_beta=selected_beta.sort_values(by=["avg"])
      first_group=selected_beta[selected_beta["Group"]==selected_beta["Group"][0]]
      if first_group.shape[0] <10:
          second_group = selected_beta[selected_beta["Group"]==selected_beta["Group"][first_group.shape[0]]]
          preselected_stock = pd.concat([first_group,second_group],axis=0,ignore_index=False)
      else:
          preselected_stock = first_group
      if preselected_stock.shape[0]>max_length:
        max_length=preselected_stock.shape[0]
      stock_list.extend(preselected_stock.index.values)
    print(stock_list)

    # Count the times each stock appear
    values,counts=np.unique(stock_list,return_counts=True)
    count_dict={}
    preselected_stock=[]
    for t in range(0,len(values)):
      count_dict[values[t]]=counts[t]
    sorted_count=sorted(count_dict.items(), key=lambda x:x[1],reverse=True)
    print(sorted_count)

    d = {'stock':[], 'frequency':[]}
    df = pd.DataFrame(data=d)
    for item in sorted_count:
      df.loc[len(df.index)] = [item[0],item[1]]
    df.to_csv('stock_frequency.csv')

    for p in range(0,min(22,len(sorted_count))):
      if sorted_count[p][1]>=0:
        preselected_stock.append(sorted_count[p][0])

    preselected_stock = sorted(preselected_stock)
    selected_beta=selected_beta.loc[preselected_stock]
    selected_beta["avg"]=selected_beta.loc[:,start_swan:end_swan].mean(axis=1)
    #selected_beta["avg"]=selected_beta["avg"].mean()
    preselected_stock = selected_beta
    print(preselected_stock)

# use dtw to cluster stocks
elif method == 'dtw':
  selected_beta =portfolio_beta.loc[:,start_swan:end_swan].dropna()

  # DTW distance matrix
  n_stocks = selected_beta.shape[0]
  dtw_dist = np.zeros((n_stocks, n_stocks))

  for i in range(n_stocks):
      ts_i = selected_beta.iloc[i].values.reshape(-1, 1)
      for j in range(i+1, n_stocks):
          ts_j = selected_beta.iloc[j].values.reshape(-1, 1)
          distance = dtw(ts_i, ts_j)
          dtw_dist[i,j] = dtw_dist[j,i] = distance if np.isfinite(distance) else 1e6
  # Validate Distance Matrix
  assert np.all(np.isfinite(dtw_dist)), "Distance matrix contains non-finite values"

  # Hierarchical Clustering
  Z = linkage(dtw_dist, method='ward', optimal_ordering=True)

  # Find iotimal cluster count
  db_scores = []
  cluster_labels = []
  possible_N = range(2,15)

  for n_clusters in possible_N:
      labels = fcluster(Z, t=n_clusters, criterion='maxclust')
      db_scores.append(davies_bouldin_score(dtw_dist, labels))
      cluster_labels.append(labels)

  optimal_idx = np.argmin(db_scores)
  optimal_N = possible_N[optimal_idx]
  labels = cluster_labels[optimal_idx]

  # Cluster evaluation
  cluster_stats = (
      selected_beta.groupby(labels)
      .mean()
      .mean(axis=1)
      .to_frame('avg_beta')
      .sort_values('avg_beta')
  )

  # Select clusters until we reach target stock count
  safe_clusters = []
  stock_count = 0

  for cluster_id in cluster_stats.index:
      cluster_mask = (labels == cluster_id)
      cluster_size = sum(cluster_mask)
      
      if stock_count + cluster_size >= selected_stock_number:
          needed = selected_stock_number - stock_count
          safe_clusters.append((cluster_id, needed))
          break
          
      safe_clusters.append((cluster_id, cluster_size))
      stock_count += cluster_size

  # Stock Selection
  preselected_stocks = []
  for cluster_id, n_stocks in safe_clusters:
      # Get stocks in cluster sorted by safety
      cluster_stocks = selected_beta.index[labels == cluster_id]
      safety_scores = selected_beta.loc[cluster_stocks].mean(axis=1)
      selected = safety_scores.nsmallest(n_stocks).index.tolist()
      
      preselected_stocks.extend(selected)

  # Output
  preselected_df = selected_beta.loc[preselected_stocks].copy()

  # Add cluster information
  cluster_map = {stock: label for stock, label in zip(selected_beta.index, labels)}
  preselected_df['Cluster'] = preselected_df.index.map(cluster_map)

  # Calculate metrics
  preselected_df['avg'] = preselected_df.mean(axis=1)
  preselected_df['PR'] = 1 - preselected_df['avg']

C:\Users\ysnow\AppData\Local\Temp\ipykernel_93552\857199420.py:127: ClusterWarning: scipy.cluster: The symmetric non-negative hollow observation matrix looks suspiciously like an uncondensed distance matrix
  Z = linkage(dtw_dist, method='ward', optimal_ordering=True)


In [26]:
preselected_df

,Swan4,Swan5,Swan6,Swan7,Swan8,Cluster,avg,PR
Ticker,,,,,,,,
SO,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1,0.202395,0.797605
KO,-0.170561,0.536675,0.325968,0.614677,-0.049527,1,0.376205,0.623795
PG,0.007947,0.845229,0.129101,1.205905,-0.307861,1,0.480054,0.519946
SBUX,0.289966,1.077850,0.586287,0.664627,-0.489111,1,0.521603,0.478397
DUK,1.580780,0.771715,-0.322795,0.233695,-0.356252,12,2.317857,-1.317857
MCD,1.261977,0.321830,0.130073,0.483323,-0.202954,12,2.332375,-1.332375
NEE,0.414536,0.503895,0.415370,0.192031,-0.026634,2,0.583200,0.416800
MDLZ,0.033697,0.828314,0.344207,0.089728,0.386481,2,0.613738,0.386262
CL,0.099786,0.617541,-0.045655,0.784583,0.569564,2,0.670970,0.329030


In [27]:
print(f"\nSelected {len(preselected_df)} stocks from:")
for cluster_id, n_stocks in safe_clusters:
    print(f"- Cluster {cluster_id}: {n_stocks} stocks")


Selected 22 stocks from:
- Cluster 1: 4 stocks
- Cluster 12: 2 stocks
- Cluster 2: 8 stocks
- Cluster 3: 5 stocks
- Cluster 4: 3 stocks


In [28]:
preselected_df.to_csv('dtw_preselected_stocks.csv')

# Step 2 - Download data and adjust price based on Swan Beta

In [29]:
preselected_stock = preselected_df

In [30]:
#next line is used to input preselected stock from excel, if you are going to preselect by your self, use this blocks and note next one.
#selected_stock_list =  pd.read_excel("19_stickers.xlsx",header = None).values[:,0]
#swan_beta=pd.read_excel("19_stickers.xlsx",header = None,index_col=0).iloc[:,0]

In [31]:
#Set the paramter to download tradeinfo from Yahoo Finance
selected_stock=preselected_stock
selected_stock_list=selected_stock.index.values
swan_beta=preselected_stock.loc[selected_stock_list,"avg"]
print(selected_stock_list)

['SO' 'KO' 'PG' 'SBUX' 'DUK' 'MCD' 'NEE' 'MDLZ' 'CL' 'EXC' 'VZ' 'PEP'
 'LLY' 'MRK' 'AMZN' 'UNH' 'SPG' 'UPS' 'LMT' 'MO' 'WMT' 'JNJ']


In [32]:
#download data from yahoo finance
yahooDataFrame= pdr.get_data_yahoo(list(selected_stock_list), start=start, end=end)["Adj Close"]

[*********************100%%**********************]  22 of 22 completed


In [33]:
# yahooDataFrame.to_csv('dtw_yahooDataFrame.csv')
yahooDataFrame=yahooDataFrame[selected_stock_list]
yahooDataFrame

Ticker,SO,KO,PG,SBUX,DUK,MCD,NEE,MDLZ,CL,EXC,...,LLY,MRK,AMZN,UNH,SPG,UPS,LMT,MO,WMT,JNJ
Date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,34.275490,38.957539,77.998978,56.432415,65.633438,153.118988,36.857197,34.759838,51.095390,25.484907,...,105.114410,60.248028,76.956497,222.475296,117.908752,79.020821,225.775497,30.678122,28.292694,108.155212
2019-01-03,34.761559,38.716801,77.452080,53.984547,65.610184,152.110123,36.766045,34.629131,50.853931,25.513819,...,101.847885,59.012634,75.014000,216.408371,117.808723,76.777252,220.104004,30.510139,28.147200,106.436569
2019-01-04,35.051628,39.488815,79.032898,55.774391,66.152603,155.049759,37.443146,35.587662,51.328217,25.802956,...,104.913109,60.790031,78.769501,218.939270,119.602257,79.451653,226.039841,31.294043,28.323008,108.222946
2019-01-07,34.973225,38.974152,78.716736,55.774391,65.873657,156.736923,37.388897,35.831654,51.638683,25.849222,...,105.480415,60.120510,81.475502,219.359634,120.859886,79.183395,228.632523,30.348381,28.656435,107.528694
2019-01-08,35.968876,39.414104,79.007278,55.818260,66.702774,157.067459,37.662350,36.668201,52.216461,26.248240,...,106.450302,60.566868,82.829002,222.292557,123.746689,79.191505,230.159164,30.522579,28.856485,110.026230
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,56.103107,51.171318,111.503174,78.602837,77.812515,189.903458,60.400391,52.441784,68.813454,28.818167,...,136.146439,70.134102,106.695503,266.692688,107.510307,86.724396,384.728485,30.682301,35.641438,132.191345
2020-02-11,56.226063,51.034286,109.726837,78.584862,78.427048,192.147949,60.491520,52.352856,67.710320,29.432590,...,134.967514,69.798409,107.540001,270.186249,106.773842,88.238945,384.807220,30.263908,35.687824,132.304474
2020-02-12,56.250656,50.880127,109.099365,79.536766,79.179085,193.688828,60.071423,52.423992,67.567978,29.384863,...,134.275101,68.160904,108.000000,281.977112,104.586967,88.802696,381.963898,30.151005,35.826992,131.521011


In [34]:
cd C:\Users\ysnow\OneDrive\Desktop\Black-Swan-Research-main\data\MVO_Components

C:\Users\ysnow\OneDrive\Desktop\Black-Swan-Research-main\data\MVO_Components


In [35]:
#set parameters to calculate predict market drop
market_drop_info = pd.read_excel("MarketDD.xlsx",index_col=0)
#swan_to_predict=100
total_day=yahooDataFrame.shape[0]
t1=int(round(market_drop_info.loc[:end_swan,"T1"].mean(),0))
t1=int((np.array(market_drop_info.loc[:end_swan,"T1"])*np.array(weight_swan)).sum())
#t2=total_day-t1
t2=int(round(market_drop_info.loc[:end_swan,"T2"].mean(),0))
dd=market_drop_info.loc[:end_swan,"Actual DD"].mean()
t0=int(total_day-t1-t2)

In [36]:
#calculate predict market drop
md = [] #Make market drop data list named md


j = 0 #Start from 0
inc = dd/(t1-1) #Calculate unit increasing market drop
for i in range(t1): #Append market drop data for t1
    md.append(round(j,4))
    j = j + inc

base = (t2+1)**(1/(dd*100)) #Calculate log base
l1 = [] #Make decreasing market drop list
for i in range(t2):
    l1.append(math.log((i+2),base)/100)

for i in range(t2): #Append market drop data for t2
    md.append(round((dd-l1[i]),4))

for i in range(t0):
    md.append(0)

market_drop = pd.DataFrame(md) #Change list into dataframe
#market_drop = market_drop.rename(columns={0: 'Swan Drop spy'}) #Change column name
market_drop1 = pd.read_csv("trainingMarketdrop.csv")

In [37]:
market_drop.to_csv("dtw_market_drop.csv")

In [38]:
market_drop

,0
0,0.0000
1,0.0037
2,0.0074
3,0.0111
4,0.0148
...,...
278,0.0000
279,0.0000
280,0.0000
281,0.0000


- Calculate Drop Impact:
Multiplies each stock’s swan beta by the predicted market drop (from market_drop) to determine the expected drop percentage for each stock. Stocks with higher beta values will have a larger predicted drop.
- Updates the prices in yahooDataFrame by applying the drop:
Adjusted Price = Original Price × (1 − Drop Percentage)
This simulates the effect of a market drop on each stock’s price.

In [39]:
# adjust stock price based on swan beta
swan_drop_pct=np.array(swan_beta)*np.array(market_drop)
yahooDataFrame=pd.DataFrame((1-swan_drop_pct)*np.array(yahooDataFrame),index=yahooDataFrame.index,columns=yahooDataFrame.columns)

In [40]:
yahooDataFrame=yahooDataFrame.dropna(how="all",axis = 1)
selected_stock_list=yahooDataFrame.columns.values
selected_stock_number = yahooDataFrame.shape[1]
selected_stock_list

array(['SO', 'KO', 'PG', 'SBUX', 'DUK', 'MCD', 'NEE', 'MDLZ', 'CL', 'EXC',
       'VZ', 'PEP', 'LLY', 'MRK', 'AMZN', 'UNH', 'SPG', 'UPS', 'LMT',
       'MO', 'WMT', 'JNJ'], dtype=object)

In [41]:
# yahooDataFrame.to_csv('dtw_yahooDataFrame.csv')
yahooDataFrame

Ticker,SO,KO,PG,SBUX,DUK,MCD,NEE,MDLZ,CL,EXC,...,LLY,MRK,AMZN,UNH,SPG,UPS,LMT,MO,WMT,JNJ
Date,,,,,,,,,,,,,,,,,,,,,
2019-01-02,34.275490,38.957539,77.998978,56.432415,65.633438,153.118988,36.857197,34.759838,51.095390,25.484907,...,105.114410,60.248028,76.956497,222.475296,117.908752,79.020821,225.775497,30.678122,28.292694,108.155212
2019-01-03,34.735527,38.662908,77.314510,53.880360,65.047506,150.797445,36.686709,34.550494,50.727682,25.449948,...,101.547086,58.829767,74.787243,215.657065,117.384459,76.484814,219.242850,30.389272,28.031263,105.995007
2019-01-04,34.999130,39.378882,78.752142,55.559110,65.017944,152.373666,37.281553,35.426035,51.073363,25.673766,...,104.293404,60.413282,78.293282,217.419086,118.740810,78.846403,224.271085,31.046096,28.089686,107.324999
2019-01-07,34.894654,38.811400,78.297286,55.451469,64.178846,152.679105,37.146859,35.587551,51.254091,25.655090,...,104.545831,59.561610,80.736634,217.074980,119.554129,78.278587,225.948958,29.987701,28.302333,106.190414
2019-01-08,35.861133,39.194653,78.445948,55.387359,64.414583,151.645625,37.337273,36.335132,51.697933,25.985402,...,105.192732,59.816136,81.827480,219.205621,121.964094,77.984970,226.557187,30.038912,28.381053,108.200411
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2020-02-10,56.103107,51.171318,111.503174,78.602837,77.812515,189.903458,60.400391,52.441784,68.813454,28.818167,...,136.146439,70.134102,106.695503,266.692688,107.510307,86.724396,384.728485,30.682301,35.641438,132.191345
2020-02-11,56.226063,51.034286,109.726837,78.584862,78.427048,192.147949,60.491520,52.352856,67.710320,29.432590,...,134.967514,69.798409,107.540001,270.186249,106.773842,88.238945,384.807220,30.263908,35.687824,132.304474
2020-02-12,56.250656,50.880127,109.099365,79.536766,79.179085,193.688828,60.071423,52.423992,67.567978,29.384863,...,134.275101,68.160904,108.000000,281.977112,104.586967,88.802696,381.963898,30.151005,35.826992,131.521011


# Step3 - MVO

In [42]:
#Set the paramter for MVO process
Portfolio_Date=end

In [43]:
#data cleaning - filling missing data in yahooDataFrame
data_cleaning(yahooDataFrame)

In [44]:
#calculated rf
#bond_yield = quandl.get('FRED/DGS3MO',api_key='X4EQzCpMfeaTCTDthRH_')
#bond_yield = quandl.get('FRED/DGS3MO')
#bond_yield=np.log(bond_yield/100+1)
#date=list(map(str,bond_yield.index.values))
#date=list(map(modify_date,date))
#week=pd.to_datetime(date).strftime("%W").values
#week_1=np.roll(week,1)
#week_difference=np.where(week!=week_1)
#week_difference=np.delete(week_difference,0,None)
#rf_week=bond_yield.iloc[week_difference-1]

calculate returns over different time horizons

In [45]:
# Calculated R for 1 year and 0.5 year
today_price = yahooDataFrame.iloc[-MA_days:,:]
one_year_ago_data = get_price_years_ago(yahooDataFrame,end,1)
one_year_ago_date = one_year_ago_data.index.values[-1]
half_year_ago_data = get_price_years_ago(yahooDataFrame,end,0.5)
half_year_ago_date = half_year_ago_data.index.values[-1]
one_year_ago_price=yahooDataFrame.loc[one_year_ago_date:,:]
one_year_ago_price=yahooDataFrame.iloc[:MA_days,:]
half_year_ago_price=yahooDataFrame.loc[half_year_ago_date:,:]
half_year_ago_price=yahooDataFrame.iloc[:MA_days,:]

# mean return estimate over the chosen window
R_mv_one_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(one_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)
R_mv_half_year=np.mean(pd.DataFrame(np.array(today_price)/np.array(half_year_ago_price)-1,index=today_price.index,columns=today_price.columns),axis=0)

In [46]:
# Calculate Moving Averages(MA) for all stocks
MA_Data=yahooDataFrame.rolling(MA_days).mean().dropna(axis=0)
MA_Data=MA_Data.loc[one_year_ago_date:,:]
MA_now = MA_Data.iloc[-1,:]
MA_one_year_ago = MA_Data.iloc[0,:]
MA_half_year_ago = MA_Data.loc[half_year_ago_date,:]

In [47]:
# Calculate daily return
daily_return = yahooDataFrame.pct_change().dropna()
log_daily_return = np.log(1+daily_return)

# Get week end stock price data
stock_date = yahooDataFrame.index.values
week=pd.to_datetime(stock_date).strftime("%W").values
week_1=np.roll(week,1)
week_difference=np.where(week!=week_1)
week_difference=np.delete(week_difference,0,None)
# Calculate weekly return

weekly_return=yahooDataFrame.iloc[week_difference-1].pct_change().dropna()
log_weekly_return = np.log(1+weekly_return)

# Get month end stock price data
stock_date = yahooDataFrame.index.values
month=pd.to_datetime(stock_date).strftime("%m").values
month_1=np.roll(month,1)
month_difference=np.where(month!=month_1)
month_difference=np.delete(month_difference,0,None)
# Calculate monthly return
monthly_return=yahooDataFrame.iloc[month_difference-1].pct_change().dropna()
log_monthly_return = np.log(1+monthly_return)

# Get semi-month end stock price data
semi_month = []
for i in range(1,len(month_difference)):
    semi_month.append((month_difference[i]-month_difference[i-1])//2+month_difference[i-1])
    semi_month.append(month_difference[i]-1)
# Calculate semi-monthly return
semi_monthly_return=yahooDataFrame.iloc[semi_month].pct_change().dropna()
log_semi_monthly_return = np.log(1+semi_monthly_return)

calculate RISK metircs: maximum drawdown and volatility

In [48]:
# Calculated Max Dropdown for each stock
stock_return_one_year=daily_return.loc[one_year_ago_date:,:]
mdd=[]
for column in stock_return_one_year.columns:
    mdd .append(max_drawdown_absolute(stock_return_one_year.loc[:,column]))
mdd=pd.DataFrame(mdd,index = stock_return_one_year.columns,columns=["MAXDD","start_index","end_index"])
mar=R_mv_one_year/mdd.iloc[:,0]

below is GARCH-Based Volatility Estimation

In [49]:
#calculate weekly sd for each stock
one_year_weekly_return=log_weekly_return.iloc[log_weekly_return.index >= one_year_ago_date]
weekly_std= np.std(one_year_weekly_return)*np.sqrt(one_year_weekly_return.shape[0])

#Calculate garch_std sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
res_all = one_year_monthly_return - np.mean(one_year_monthly_return)
judge = one_year_monthly_return.shape[0]
for stock in one_year_monthly_return.columns:
    res=res_all[stock]
    h= het_arch(res)
    if judge < 100 or h ==0:
        garch_std = weekly_std

    else:#build Garch Model
        std = []
        if judge <=200:
            p = 0
            q = 6
        else:
            p = 1
            q = 1

        std.append(fit_garch(one_year_monthly_return[stock],p,q))
        garch_std = pd.DataFrame(std,index = one_year_monthly_return.columns)

#Calculate monthly sd for each stock
one_year_monthly_return=log_monthly_return.iloc[log_monthly_return.index >= one_year_ago_date]
monthly_std = np.std(one_year_monthly_return)*np.sqrt(12);

#Calculate semi_monthly sd for each stock
one_year_semi_monthly_return=log_semi_monthly_return.iloc[log_semi_monthly_return.index >= one_year_ago_date]
semi_monthly_std = np.std(one_year_semi_monthly_return)*np.sqrt(24);


c:\Users\ysnow\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


In [50]:
# Calculated MaxDrop
DD_range = 90
total_range = 90
DD_end = yahooDataFrame.iloc[yahooDataFrame.index <= one_year_ago_date]
DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
max_DD=[-1]*selected_stock_number


if (DD_start.shape[0] != 1 ) and (DD_end.shape[0] != 1):
    DD_start = DD_start.index[-1]
    DD_end = DD_end.index[-1]
    while DD_start > DD_end :
        DD_start = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(total_range)]
        DD_mid = yahooDataFrame.iloc[yahooDataFrame.index <= end-datetime.timedelta(DD_range+total_range)]
        if (DD_start.shape[0] != 1 ) and (DD_mid.shape[0] != 1):
            DD_start = DD_start.index[-1]
            DD_mid = DD_mid.index[-1]
            price = yahooDataFrame.loc[DD_start:DD_mid,:]
            for i in range(len(yahooDataFrame.columns)):
                max_DD[i]=max(max_DD[i],max_drawdown_absolute(stock_return_one_year.iloc[:,i])[0])
            total_range = total_range + 1
        else:
            break

incorperate forward-looking estimates and aggregate metrics

In [51]:
cd C:\Users\ysnow\OneDrive\Desktop\Black-Swan-Research-main\data\optdata

C:\Users\ysnow\OneDrive\Desktop\Black-Swan-Research-main\data\optdata


In [52]:
# Read OPT-data
optdata = pd.read_excel("optdata_MVO2.xlsx",index_col=0,sheet_name = 1)
P_current = MA_now
P_fwd = optdata.P_fwd.loc[P_current.index]
R_fwd=P_fwd/P_current-1
R_60 = P_current/MA_one_year_ago - 1
R_60_half = P_current/MA_half_year_ago - 1
R_MA_1ago = R_mv_one_year
R_MA_halfago = R_mv_half_year
Rfwd_adjusted = pd.Series(index=R_MA_1ago.index)
for i in R_fwd.index:
    if R_fwd[i] < 0.5*R_60[i]:
        Rfwd_adjusted[i] = R_60_half[i]
    elif R_fwd[i] > 1.3*R_60[i]:
        if 1.3*R_60[i] > 0.7*R_fwd[i]:
            Rfwd_adjusted[i] = 1.3*R_60[i]
        else:
            Rfwd_adjusted[i] = 0.7*R_fwd[i]
    else:
        Rfwd_adjusted[i] = R_fwd[i]
Rfwd_adjusted[Rfwd_adjusted<0]=0

In [53]:
price_now = today_price.iloc[-1,:]
optimizer = optdata.loc[P_current.index,:]
FFC = pd.DataFrame([R_MA_1ago.values,R_MA_halfago.values,MA_now.values,price_now.values,semi_monthly_std.values,weekly_std.values,MA_one_year_ago.values,MA_half_year_ago.values,max_DD,R_60.values,R_60_half.values,P_fwd.values,Rfwd_adjusted.values],columns = R_MA_1ago.index,index = ["R_MA_1ago","R_MA_halfago","MA_now","price",'std','std_f','MA_1ago','MA_halfago','Fall_MDD','R_60','R_60_half','R_fwd',"Rfwd_adjusted"]).T
preselected_stock = preselected_stock.loc[selected_stock_list,:]
preselected_stock["Rfwd_adjusted"]=Rfwd_adjusted.values
preselected_stock["std_f"]=weekly_std.values
preselected_stock["SR"]=preselected_stock["Rfwd_adjusted"]/preselected_stock["std_f"]
preselected_stock["PR_SR"]=preselected_stock["PR"]*preselected_stock["SR"]**0.5
# preselected_stock.to_csv("preselected_Stocks_info.csv")
p=FFC.MA_now
EPS_fwd=optimizer.EPS_fwd
EPS_ttm=optimizer.EPS_ttm
fall=FFC.Fall_MDD
# FFC.to_csv("WEDataBank.csv")

PE=p/EPS_fwd
PEG= p/(EPS_fwd*(EPS_fwd/EPS_ttm-1))
PEG[PEG<0]=2000


In [54]:
cov = pd.DataFrame(np.cov(np.exp(one_year_weekly_return)-1,rowvar=False,bias = False)*51,index= one_year_weekly_return.columns, columns= one_year_weekly_return.columns)
E_2 = Rfwd_adjusted/fall
E_3 = Rfwd_adjusted/PEG
E_4 = Rfwd_adjusted
std =  np.std(np.exp(one_year_weekly_return)-1)*np.sqrt(52)


c:\Users\ysnow\anaconda3\Lib\site-packages\numpy\core\fromnumeric.py:3643: FutureWarning: The behavior of DataFrame.std with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return std(axis=axis, dtype=dtype, out=out, ddof=ddof, **kwargs)


# Optimization

In [55]:
w0 = 1/selected_stock_number*np.ones(selected_stock_number)
uplim_ini = np.zeros(selected_stock_number)
uplim3 =pd.DataFrame( w1*(1-uplim_ini) + w2 * uplim_ini,index = std.index,columns=["weight"])
for i in uplim3.index:
    if i in ["SPY","IWD"]:
        uplim3.loc[i,:]=0
low_lim =  np.zeros(selected_stock_number)
multi=lambda x: x.T @cov@x

In [56]:
ef = EfficientFrontier(E_2/100, cov,weight_bounds=(0, w1))
#ef.add_constraint(lambda w: w@E_2== 12)
raw_weights = ef.max_sharpe(0.01)
cleaned_weights = ef.clean_weights()
print(cleaned_weights)
ef.portfolio_performance(verbose=True)


OrderedDict({'SO': 0.17, 'KO': 0.04133, 'PG': 0.07626, 'SBUX': 0.0, 'DUK': 0.0, 'MCD': 0.0, 'NEE': 0.17, 'MDLZ': 0.0, 'CL': 0.0, 'EXC': 0.0, 'VZ': 0.0, 'PEP': 0.0, 'LLY': 0.14788, 'MRK': 0.01528, 'AMZN': 0.17, 'UNH': 0.00863, 'SPG': 0.0, 'UPS': 0.03522, 'LMT': 0.0, 'MO': 0.01393, 'WMT': 0.15147, 'JNJ': 0.0})
Expected annual return: 30.3%
Annual volatility: 9.6%
Sharpe Ratio: 3.04


c:\Users\ysnow\anaconda3\Lib\site-packages\pypfopt\efficient_frontier\efficient_frontier.py:441: UserWarning: The risk_free_rate provided to portfolio_performance is different to the one used by max_sharpe. Using the previous value.
  warnings.warn(


(0.30262419454394585, 0.09636840483174973, 3.0365159105293946)

In [57]:
weight1=np.array(pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"]))[:,0]

In [58]:
sd1 = np.sqrt(weight1@cov@weight1.T)
R1 = weight1.T@E_2/100
sharpe_ratio1 = (R1-0.01)/sd1
print("new_sharpe:",sharpe_ratio1)

new_sharpe: 3.036515910219019


weight2 =   pd.read_excel("old_weight.xlsx",index_col=0)
weight2=np.array(weight2.loc[list(cleaned_weights),:])[:,0]

sd2 = np.sqrt(weight2@cov@weight2.T)
R2 = weight2.T@E_2/100
sharpe_ratio2 = (R2-0.01)/sd2
print("old_sharpe:",sharpe_ratio2)

In [59]:
weight=pd.DataFrame(cleaned_weights.values(), index = list(cleaned_weights),columns = ["weight"])
weight = weight[weight.weight >=0.030]
weight.weight = weight.weight/sum(weight.weight)
#weight.to_csv("weights.csv")

See pyportfolioopt.readthedocs.io/en/latest/MeanVariance.html for more reference


In [60]:
if stock_type == "S&P100":
    OEX=pdr.get_data_yahoo("^OEX", start=start_swan_date, end=end_swan_date)["Adj Close"]

elif stock_type == "S&P500":
    OEX=pdr.get_data_yahoo("SPY", start=start_swan_date, end=end_swan_date)["Adj Close"]

stock_price=pdr.get_data_yahoo(list(weight.index), start=start_swan_date, end=end_swan_date)["Adj Close"]

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed


In [61]:
# stock_price.to_csv("tradeinfo.csv")
stock_price=stock_price[weight.index]
stock_price

Ticker,SO,KO,PG,NEE,LLY,AMZN,UPS,WMT
Date,,,,,,,,
2020-02-18,57.401829,50.982903,110.354317,62.689789,132.619858,107.783501,89.543121,36.995964
2020-02-19,56.748413,51.188442,110.858040,61.051636,132.450562,108.511002,89.829201,36.392921
2020-02-20,57.740955,51.145622,111.865517,61.576210,133.550949,107.654999,89.593620,36.396004
2020-02-21,57.062721,51.496754,111.971581,61.700680,133.588577,104.798500,86.615021,36.671246
2020-02-24,56.078453,50.229240,108.966820,60.889393,129.930008,100.464500,83.691856,35.972340
...,...,...,...,...,...,...,...,...
2021-03-16,52.944157,45.744759,116.776077,68.760605,182.958145,154.593002,142.452240,41.939919
2021-03-17,52.468739,45.762615,116.305161,66.553116,178.961639,156.786499,140.553696,41.590904
2021-03-18,52.209419,45.164230,116.558731,64.883804,176.278122,151.399506,140.160004,41.047867


In [62]:
weight

,weight
SO,0.176686
KO,0.042955
PG,0.079259
NEE,0.176686
LLY,0.153696
AMZN,0.176686
UPS,0.036605
WMT,0.157427


In [63]:
start=stock_price.loc["2020-02-19"]
end = stock_price.loc["2020-03-23"]
stock_drop=abs(end/start-1)
market_drop = abs(OEX.loc["2020-03-23"]/OEX.loc["2020-02-19"]-1)

In [64]:
beta = stock_drop/market_drop
1-beta.T@weight

weight    0.32757
dtype: float64

In [65]:
beta

Ticker
SO      1.173178
KO      1.161008
PG      0.701342
NEE     1.062980
LLY     0.490481
AMZN    0.390752
UPS     0.414386
WMT     0.078002
dtype: float64

In [66]:
stock_drop

Ticker
SO      0.369917
KO      0.366080
PG      0.221141
NEE     0.335170
LLY     0.154654
AMZN    0.123209
UPS     0.130661
WMT     0.024595
dtype: float64

In [67]:
rest=1
extra=0
for i, row in weight.iterrows():
  if row['weight']>0.17:
    rest=rest-row['weight']
    extra=extra+(row['weight']-0.17)
    # row['weight']=0.17

for i,row in weight.iterrows():
  if row['weight']<0.165:
    # row['weight']=row['weight']+row['weight']/rest*extra
    a=1

# weight.to_csv("weights.csv")
weight

,weight
SO,0.176686
KO,0.042955
PG,0.079259
NEE,0.176686
LLY,0.153696
AMZN,0.176686
UPS,0.036605
WMT,0.157427


In [68]:
portfolio_beta = portfolio_beta.loc[weight.index]
portfolio_beta

,Swan -4,Swan -3,Swan -2,Swan -1,Swan1,Swan2,Swan3,Swan4,Swan5,Swan6,Swan7,Swan8,Swan9,Swan10D
SO,NaN,NaN,NaN,0.789771,0.735961,-0.130115,-0.265152,-0.138738,0.669546,-0.279809,0.055185,-0.091814,1.173178,0.264469
KO,1.608792,0.722924,-0.425107,-0.548372,1.142372,0.525785,1.283609,-0.170561,0.536675,0.325968,0.614677,-0.049527,1.161009,0.181359
PG,0.705288,0.818261,-1.019843,0.406437,1.140579,0.904186,0.799958,0.007947,0.845229,0.129101,1.205905,-0.307861,0.701342,0.817893
NEE,0.982309,0.253298,-0.134131,0.609003,0.579708,0.166846,-0.420656,0.414536,0.503895,0.415370,0.192031,-0.026634,1.062981,0.710514
LLY,0.500964,0.963242,0.201970,-0.391697,1.143232,0.898002,0.125930,0.746269,0.920577,0.280674,0.780346,0.061464,0.490481,-0.761718
AMZN,NaN,NaN,NaN,NaN,NaN,NaN,1.584517,-0.445606,0.328167,0.123350,0.334033,1.561997,0.390752,1.239298
UPS,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.004962,0.930849,0.907385,0.188714,1.154634,0.414387,0.864684
WMT,0.674982,0.897880,-1.424143,0.527944,1.059816,1.484267,0.762945,0.586029,0.461023,0.235665,0.982187,0.414453,0.078003,0.302591


In [69]:
a=stock_price@weight
b=1-a.loc["2020-03-23"]/a.loc["2020-02-19"]
1-b/market_drop

weight    0.362376
dtype: float64

In [70]:
injected_result = pd.DataFrame(index = stock_price.index)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T


In [71]:
portfolio_value = np.sum(np.array(weight.T) * stock_price,axis=1)
S1 = np.array(10000*weight.T)/np.array(stock_price.iloc[0,:]).T
injected_result["portfolio_value"]=stock_price@S1.T
S2 = 10000/float(OEX[0])
injected_result["oex"]=S2*OEX
injected_result.to_csv("dtw_injected_result.csv")

C:\Users\ysnow\AppData\Local\Temp\ipykernel_93552\1565572301.py:4: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])


In [72]:
injected_result

,portfolio_value,oex
Date,,
2020-02-18,10000.000000,10000.000000
2020-02-19,9924.539408,10051.749340
2020-02-20,9974.638777,10000.992803
2020-02-21,9914.136894,9885.978741
2020-02-24,9673.579152,9531.473020
...,...,...
2021-03-16,11813.262437,11894.422815
2021-03-17,11700.209113,11926.121417
2021-03-18,11497.834591,11731.364853


In [73]:
temp=injected_result.loc[["2020-02-19","2020-03-23"]]
Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
DD=abs(temp['oex'][1]/temp['oex'][0]-1)
PR=1-(Portfolio_DD/DD)
print('DD', DD)
print('Portfolio_DD', Portfolio_DD)
print('PR', PR)
temp

DD 0.3153119674783472
Portfolio_DD 0.21154224909087993
PR 0.32910174395646197


C:\Users\ysnow\AppData\Local\Temp\ipykernel_93552\833904749.py:2: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  Portfolio_DD=abs(temp['portfolio_value'][1]/temp['portfolio_value'][0]-1)
C:\Users\ysnow\AppData\Local\Temp\ipykernel_93552\833904749.py:3: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  DD=abs(temp['oex'][1]/temp['oex'][0]-1)


,portfolio_value,oex
Date,,
2020-02-19,9924.539408,10051.749340
2020-03-23,7825.080020,6882.312479


In [209]:
cd C:\Users\ysnow\OneDrive\Desktop\Black-Swan-Research-main\data\MVO_Components

C:\Users\ysnow\OneDrive\Desktop\Black-Swan-Research-main\data\MVO_Components


In [210]:
allocation=pd.read_excel('allocation weight.xlsx',index_col='stock selected')
dfMVOstock=deepcopy(weight)
dfMVOstock['PR_SR']=''
dfMVOstock['SR']=''
for i in range(len(weight)):
    dfMVOstock.iloc[i,1]=preselected_stock.loc[weight.index[i],'PR_SR']
    dfMVOstock.iloc[i,2]=preselected_stock.loc[weight.index[i],'SR']
dfMVOstock_sort=dfMVOstock.sort_values(by='PR_SR',ascending=False)
postallocationweight=post_allocation(dfMVOstock_sort)
postallocationweight

C:\Users\ysnow\AppData\Local\Temp\ipykernel_30344\2775426837.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  if (dfMVOweight.iloc[i,4]+residual/(numofstocks-i))>allocationweight[0]:
C:\Users\ysnow\AppData\Local\Temp\ipykernel_30344\2775426837.py:13: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  dfMVOweight.iloc[i,5]=allocationweight[0]
C:\Users\ysnow\AppData\Local\Temp\ipykernel_30344\2775426837.py:12: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, u

,weight,PR_SR,SR,equal,proportion,adjusted proportion,direct
NEE,0.176686,1.739437,17.416524,0.125,0.322879,0.17,0.17
AMZN,0.176686,1.67018,83.286887,0.125,0.310023,0.17,0.17
SO,0.176686,1.077498,1.824978,0.125,0.200008,0.17,0.17
KO,0.042955,0.541227,0.752792,0.125,0.100464,0.165046,0.17
PG,0.079259,0.475349,0.835812,0.125,0.088235,0.152817,0.17
LLY,0.153696,0.258907,1.64641,0.125,0.048059,0.112641,0.05
UPS,0.036605,-0.021819,0.549414,0.125,-0.00405,0.060532,0.05
WMT,0.157427,-0.3535,9.747343,0.125,-0.065618,-0.001036,0.05


In [ ]:
#draw type E graph for postallocation
MVOweight=postallocationweight['weight']
equalweight=postallocationweight['equal']
proportionweight=postallocationweight['adjusted proportion']
directweight=postallocationweight['direct']
output(MVOweight).to_csv('no_post_allocation_result.csv')
output(equalweight).to_csv('equal_weight_result.csv')
output(proportionweight).to_csv('proportional_weight_result.csv')
output(directweight).to_csv('direct_weight_result.csv')

[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-12-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 completed
<ipython-input-12-01d80541063f>:14: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  S2 = 10000/float(OEX[0])
[*********************100%%**********************]  1 of 1 completed
[*********************100%%**********************]  8 of 8 comple

#